In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

# 사용자 기본 특성

In [3]:
PROJECT_ROOT = Path.cwd().parents[1]

user_df = pd.read_csv(PROJECT_ROOT / "data" / "interim" / "accounts_user_preprocessed.csv")

user_df.head()

,id,gender,point,is_push_on,created_at,ban_status,report_count,alarm_count,pending_chat,pending_votes,group_id
0,831962,F,2248,1,2023-03-29 05:18:56.162368,N,253,40878,5499,110,12.0
1,832151,M,1519,0,2023-03-29 12:56:34.989468,N,0,37,0,47,1.0
2,832340,F,57,1,2023-03-29 12:56:35.020790,N,0,19,0,21,1.0
3,832520,M,1039,0,2023-03-29 12:56:35.049311,N,0,29,0,15,12.0
4,832614,M,1048,1,2023-03-29 12:56:35.064406,N,0,28,0,14,12.0


In [4]:
user_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 677081 entries, 0 to 677080
Data columns (total 11 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   id             677081 non-null  int64  
 1   gender         677081 non-null  str    
 2   point          677081 non-null  int64  
 3   is_push_on     677081 non-null  int64  
 4   created_at     677081 non-null  str    
 5   ban_status     677081 non-null  str    
 6   report_count   677081 non-null  int64  
 7   alarm_count    677081 non-null  int64  
 8   pending_chat   677081 non-null  int64  
 9   pending_votes  677081 non-null  int64  
 10  group_id       677080 non-null  float64
dtypes: float64(1), int64(7), str(3)
memory usage: 56.8 MB


In [5]:
user_df[user_df['group_id'].isna()]

,id,gender,point,is_push_on,created_at,ban_status,report_count,alarm_count,pending_chat,pending_votes,group_id
138958,995177,F,600,1,2023-05-08 07:25:51.737820,N,0,0,0,0,NaN


In [6]:
user_df = user_df[~user_df['group_id'].isna()]
user_df.info()

<class 'pandas.DataFrame'>
Index: 677080 entries, 0 to 677080
Data columns (total 11 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   id             677080 non-null  int64  
 1   gender         677080 non-null  str    
 2   point          677080 non-null  int64  
 3   is_push_on     677080 non-null  int64  
 4   created_at     677080 non-null  str    
 5   ban_status     677080 non-null  str    
 6   report_count   677080 non-null  int64  
 7   alarm_count    677080 non-null  int64  
 8   pending_chat   677080 non-null  int64  
 9   pending_votes  677080 non-null  int64  
 10  group_id       677080 non-null  float64
dtypes: float64(1), int64(7), str(3)
memory usage: 62.0 MB


In [7]:
# 가입일 범위
print("가입일 최소:", user_df["created_at"].min())
print("가입일 최대:", user_df["created_at"].max())

# 성별 분포
print("\n[gender]")
print(user_df["gender"].value_counts(dropna=False))

# push 설정
print("\n[is_push_on]")
print(user_df["is_push_on"].value_counts(dropna=False))

# ban 상태
print("\n[ban_status]")
print(user_df["ban_status"].value_counts(dropna=False))

가입일 최소: 2023-03-29 05:18:56.162368
가입일 최대: 2024-05-09 08:31:17.710824

[gender]
gender
F    396664
M    280416
Name: count, dtype: int64

[is_push_on]
is_push_on
1    570846
0    106234
Name: count, dtype: int64

[ban_status]
ban_status
N     668427
W       7855
NB       608
RB       190
Name: count, dtype: int64


In [8]:
activity_cols = [
    "point",
    "alarm_count",
    "pending_chat",
    "pending_votes",
    "report_count"
]

user_df[activity_cols].describe()

,point,alarm_count,pending_chat,pending_votes,report_count
count,6.770800e+05,677080.000000,677080.000000,677080.000000,677080.000000
mean,3.039165e+03,0.946653,0.098477,84.631143,0.037291
std,1.076024e+06,56.115065,11.087809,123.262928,0.588107
min,0.000000e+00,0.000000,0.000000,0.000000,0.000000
25%,4.000000e+02,0.000000,0.000000,2.000000,0.000000
50%,9.650000e+02,1.000000,0.000000,29.000000,0.000000
75%,2.183000e+03,1.000000,0.000000,122.000000,0.000000
max,8.850000e+08,40878.000000,5712.000000,3352.000000,253.000000


### accounts_user 1차 확인 메모
- 가입일 범위는 2023-03-29 ~ 2024-05-09로 확인되어, Hackle 이벤트 수집 기간인 2023-07-18 ~ 2023-08-10보다 훨씬 긴 기간을 포함하고 있음
- 따라서 이후 Hackle 행동 데이터와 직접 비교하는 분석에서는 분석 기간을 별도로 맞출 필요가 있음
- `point` 컬럼은 중앙값과 비교해 매우 큰 최대값이 확인되어 극단값 후보가 존재함
- 현재 단계는 전체 행동 루프를 빠르게 탐색하는 1차 EDA이므로, 해당 이상값은 즉시 제거하지 않고 추가 확인이 필요한 항목으로만 기록

---


# 친구 연결

In [9]:
friend_df = pd.read_csv(
    PROJECT_ROOT / "data" / "interim" / "accounts_friendrequest_preprocessed.csv"
)

contacts_df = pd.read_csv(
    PROJECT_ROOT / "data" / "interim" / "accounts_user_contacts_preprocessed.csv"
)

In [10]:
print("=== accounts_friendrequest ===")
print("행/열:", friend_df.shape)
friend_df.info()

print("\n=== accounts_user_contacts ===")
print("행/열:", contacts_df.shape)
contacts_df.info()

=== accounts_friendrequest ===
행/열: (17147175, 7)
<class 'pandas.DataFrame'>
RangeIndex: 17147175 entries, 0 to 17147174
Data columns (total 7 columns):
 #   Column           Dtype
---  ------           -----
 0   id               int64
 1   status           str  
 2   created_at       str  
 3   updated_at       str  
 4   receive_user_id  int64
 5   send_user_id     int64
 6   user_id          int64
dtypes: int64(4), str(3)
memory usage: 915.8 MB

=== accounts_user_contacts ===
행/열: (5063, 4)
<class 'pandas.DataFrame'>
RangeIndex: 5063 entries, 0 to 5062
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   id                   5063 non-null   int64
 1   contacts_count       5063 non-null   int64
 2   invite_user_id_list  5063 non-null   str  
 3   user_id              5063 non-null   int64
dtypes: int64(3), str(1)
memory usage: 158.3 KB


In [11]:
print(friend_df.columns.tolist())
print(contacts_df.columns.tolist())

['id', 'status', 'created_at', 'updated_at', 'receive_user_id', 'send_user_id', 'user_id']
['id', 'contacts_count', 'invite_user_id_list', 'user_id']


In [12]:
friend_df.head()

,id,status,created_at,updated_at,receive_user_id,send_user_id,user_id
0,7,P,2023-04-17 18:29:11,2023-04-17 18:29:11,831962,837521,837521
1,10,A,2023-04-17 18:29:11,2023-04-22 06:02:53,832151,837521,837521
2,11,A,2023-04-17 18:29:11,2023-04-18 19:28:41,832340,837521,837521
3,13,A,2023-04-17 18:29:11,2023-04-19 11:05:04,833041,837521,837521
4,20,P,2023-04-17 18:29:11,2023-04-17 18:29:11,834415,837521,837521


In [13]:
friend_df["status"].value_counts(dropna=False)

status
A    12878407
P     3938608
R      330160
Name: count, dtype: int64

In [14]:
friend_df["created_at"] = pd.to_datetime(
    friend_df["created_at"],
    errors="coerce"
)

friend_df["updated_at"] = pd.to_datetime(
    friend_df["updated_at"],
    errors="coerce"
)

In [15]:
sent_summary = (
    friend_df
    .groupby("send_user_id")
    .size()
    .reset_index(name="sent_request_count")
    .rename(columns={"send_user_id": "user_id"})
)

sent_summary.head()

,user_id,sent_request_count
0,831962,1
1,832151,10
2,832340,26
3,833024,2
4,833041,35


In [16]:
received_summary = (
    friend_df
    .groupby("receive_user_id")
    .size()
    .reset_index(name="received_request_count")
    .rename(columns={"receive_user_id": "user_id"})
)

received_summary.head()

,user_id,received_request_count
0,831962,26
1,832151,35
2,832340,25
3,832520,14
4,832614,12


In [17]:
friend_summary = sent_summary.merge(
    received_summary,
    on="user_id",
    how="outer"
)

friend_summary.head()

,user_id,sent_request_count,received_request_count
0,831962,1.0,26.0
1,832151,10.0,35.0
2,832340,26.0,25.0
3,832520,NaN,14.0
4,832614,NaN,12.0


In [18]:
friend_summary[
    ["sent_request_count", "received_request_count"]
] = friend_summary[
    ["sent_request_count", "received_request_count"]
].fillna(0).astype(int)

In [19]:
friend_summary.head()

,user_id,sent_request_count,received_request_count
0,831962,1,26
1,832151,10,35
2,832340,26,25
3,832520,0,14
4,832614,0,12


In [20]:
sent_status = (
    friend_df
    .groupby(["send_user_id", "status"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
    .rename(columns={
        "send_user_id": "user_id",
        "A": "sent_accepted_count",
        "P": "sent_pending_count",
        "R": "sent_rejected_count"
    })
)

sent_status.head()

status,user_id,sent_accepted_count,sent_pending_count,sent_rejected_count
0,831962,1,0,0
1,832151,2,8,0
2,832340,7,19,0
3,833024,1,1,0
4,833041,8,27,0


In [21]:
received_status = (
    friend_df
    .groupby(["receive_user_id", "status"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
    .rename(columns={
        "receive_user_id": "user_id",
        "A": "received_accepted_count",
        "P": "received_pending_count",
        "R": "received_rejected_count"
    })
)

received_status.head()

status,user_id,received_accepted_count,received_pending_count,received_rejected_count
0,831962,0,26,0
1,832151,6,29,0
2,832340,4,21,0
3,832520,0,14,0
4,832614,0,12,0


In [22]:
friend_summary = (
    friend_summary
    .merge(sent_status, on="user_id", how="left")
    .merge(received_status, on="user_id", how="left")
)

status_cols = [
    "sent_accepted_count",
    "sent_pending_count",
    "sent_rejected_count",
    "received_accepted_count",
    "received_pending_count",
    "received_rejected_count",
]

friend_summary[status_cols] = (
    friend_summary[status_cols]
    .fillna(0)
    .astype(int)
)

friend_summary.head()

,user_id,sent_request_count,received_request_count,sent_accepted_count,sent_pending_count,sent_rejected_count,received_accepted_count,received_pending_count,received_rejected_count
0,831962,1,26,1,0,0,0,26,0
1,832151,10,35,2,8,0,6,29,0
2,832340,26,25,7,19,0,4,21,0
3,832520,0,14,0,0,0,0,14,0
4,832614,0,12,0,0,0,0,12,0


In [23]:
friend_summary.describe()

,user_id,sent_request_count,received_request_count,sent_accepted_count,sent_pending_count,sent_rejected_count,received_accepted_count,received_pending_count,received_rejected_count
count,6.737050e+05,673705.000000,673705.000000,673705.000000,673705.000000,673705.000000,673705.000000,673705.000000,673705.000000
mean,1.211877e+06,25.452052,25.452052,19.115795,5.846191,0.490066,19.115795,5.846191,0.490066
std,2.134849e+05,23.909896,19.606964,16.819279,7.893740,2.381075,18.624269,11.136307,1.914936
min,8.319620e+05,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.027320e+06,10.000000,10.000000,8.000000,1.000000,0.000000,3.000000,0.000000,0.000000
50%,1.210390e+06,20.000000,22.000000,15.000000,4.000000,0.000000,14.000000,1.000000,0.000000
75%,1.396415e+06,34.000000,37.000000,26.000000,8.000000,0.000000,30.000000,6.000000,0.000000
max,1.583732e+06,1816.000000,218.000000,654.000000,968.000000,452.000000,216.000000,166.000000,89.000000


### accounts_friendrequest 사용자별 친구 관계 요약

- 사용자별 친구 요청 수는 평균적으로 발신·수신 모두 약 25.45건 수준으로 나타남
- 중앙값은 보낸 요청 20건, 받은 요청 22건으로 확인됨
- 친구 요청 상태는 전반적으로 `Accepted` 비중이 높고, `Rejected`는 대부분의 사용자에서 거의 발생하지 않음
- 보낸 요청 기준 중앙값은 `Accepted` 15건, `Pending` 4건, `Rejected` 0건
- 받은 요청 기준 중앙값은 `Accepted` 14건, `Pending` 1건, `Rejected` 0건
- 일부 사용자에서는 친구 요청 수가 매우 크게 나타나며, 특히 보낸 요청 최대값이 1,816건으로 확인되어 극단적 활동 사용자 후보가 존재함
- 현재 단계에서는 이상 사용자를 제거하지 않고, 친구 연결 활동량의 분포 특성으로만 기록

---

# 질문 참여·투표

In [24]:
question_record_df = pd.read_csv(
    PROJECT_ROOT / "data" / "interim" / "accounts_userquestionrecord_preprocessed.csv"
)

question_df = pd.read_csv(
    PROJECT_ROOT / "data" / "interim" / "polls_question_preprocessed.csv"
)

question_piece_df = pd.read_csv(
    PROJECT_ROOT / "data" / "interim" / "polls_questionpiece_preprocessed.csv"
)

question_set_df = pd.read_csv(
    PROJECT_ROOT / "data" / "interim" / "polls_questionset_preprocessed.csv"
)

In [25]:
for name, df in {
    "accounts_userquestionrecord": question_record_df,
    "polls_question": question_df,
    "polls_questionpiece": question_piece_df,
    "polls_questionset": question_set_df,
}.items():
    print(f"\n=== {name} ===")
    print("행/열:", df.shape)
    print(df.columns.tolist())


=== accounts_userquestionrecord ===
행/열: (1217558, 12)
['id', 'status', 'created_at', 'chosen_user_id', 'question_id', 'user_id', 'question_piece_id', 'has_read', 'answer_status', 'answer_updated_at', 'report_count', 'opened_times']

=== polls_question ===
행/열: (5025, 3)
['id', 'question_text', 'created_at']

=== polls_questionpiece ===
행/열: (1264349, 5)
['id', 'is_voted', 'created_at', 'question_id', 'is_skipped']

=== polls_questionset ===
행/열: (157705, 6)
['id', 'question_piece_id_list', 'opening_time', 'status', 'created_at', 'user_id']


In [26]:
# ① 친구 연결 구간 핵심 수치

total_requests = len(friend_df)

status_counts = friend_df["status"].value_counts()
status_rates = friend_df["status"].value_counts(normalize=True) * 100

print("=== 전체 친구 요청 상태 ===")
for status in ["A", "P", "R"]:
    print(
        status,
        f"{status_counts.get(status, 0):,}건",
        f"({status_rates.get(status, 0):.2f}%)"
    )

# 사용자별 친구 연결 성공 여부
friend_summary["has_accepted_friend"] = (
    (friend_summary["sent_accepted_count"] > 0) |
    (friend_summary["received_accepted_count"] > 0)
)

friend_summary["has_pending"] = (
    (friend_summary["sent_pending_count"] > 0) |
    (friend_summary["received_pending_count"] > 0)
)

friend_summary["total_accepted"] = (
    friend_summary["sent_accepted_count"]
    + friend_summary["received_accepted_count"]
)

friend_summary["total_pending"] = (
    friend_summary["sent_pending_count"]
    + friend_summary["received_pending_count"]
)

print("\n=== 사용자 기준 ===")
print("친구요청 사용자 수:", f"{len(friend_summary):,}")
print(
    "수락된 관계가 1건도 없는 사용자:",
    f"{(~friend_summary['has_accepted_friend']).sum():,}",
    f"({(~friend_summary['has_accepted_friend']).mean()*100:.2f}%)"
)
print(
    "Pending이 1건 이상 있는 사용자:",
    f"{friend_summary['has_pending'].sum():,}",
    f"({friend_summary['has_pending'].mean()*100:.2f}%)"
)

print("\n수락 건수 중앙값:", friend_summary["total_accepted"].median())
print("Pending 건수 중앙값:", friend_summary["total_pending"].median())

=== 전체 친구 요청 상태 ===
A 12,878,407건 (75.11%)
P 3,938,608건 (22.97%)
R 330,160건 (1.93%)

=== 사용자 기준 ===
친구요청 사용자 수: 673,705
수락된 관계가 1건도 없는 사용자: 23,426 (3.48%)
Pending이 1건 이상 있는 사용자: 651,199 (96.66%)

수락 건수 중앙값: 34.0
Pending 건수 중앙값: 8.0


In [27]:
# ② 질문 참여·콘텐츠 구간 구조 확인

print("=== accounts_userquestionrecord ===")

for col in ["status", "has_read", "answer_status"]:
    print(f"\n[{col}]")
    print(question_record_df[col].value_counts(dropna=False))
    print(
        question_record_df[col]
        .value_counts(normalize=True, dropna=False)
        .mul(100)
        .round(2)
    )

print("\n[opened_times]")
print(question_record_df["opened_times"].describe())

print("\n=== polls_questionpiece ===")

for col in ["is_voted", "is_skipped"]:
    print(f"\n[{col}]")
    print(question_piece_df[col].value_counts(dropna=False))
    print(
        question_piece_df[col]
        .value_counts(normalize=True, dropna=False)
        .mul(100)
        .round(2)
    )

=== accounts_userquestionrecord ===

[status]
status
C    1156322
I      60578
B        658
Name: count, dtype: int64
status
C    94.97
I     4.98
B     0.05
Name: proportion, dtype: float64

[has_read]
has_read
1    675931
0    541627
Name: count, dtype: int64
has_read
1    55.52
0    44.48
Name: proportion, dtype: float64

[answer_status]
answer_status
N    1097932
A     111761
P       7865
Name: count, dtype: int64
answer_status
N    90.17
A     9.18
P     0.65
Name: proportion, dtype: float64

[opened_times]
count    1.217558e+06
mean     6.332511e-02
std      3.011468e-01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      3.000000e+00
Name: opened_times, dtype: float64

=== polls_questionpiece ===

[is_voted]
is_voted
1    1217560
0      46789
Name: count, dtype: int64
is_voted
1    96.3
0     3.7
Name: proportion, dtype: float64

[is_skipped]
is_skipped
0    1264349
Name: count, dtype: int64
is_skipped
0    100.0
Name: proportion, dty

In [28]:
# pending_votes 분포 확인
print("=== pending_votes 기본 통계 ===")
print(user_df["pending_votes"].describe())

print("\n=== pending_votes 주요 분위수 ===")
print(
    user_df["pending_votes"]
    .quantile([0.5, 0.75, 0.9, 0.95, 0.99])
)

# 구간별 사용자 수 / 비율
bins = [-1, 0, 20, 100, 500, float("inf")]
labels = ["0", "1-20", "21-100", "101-500", "500+"]

pending_group = pd.cut(
    user_df["pending_votes"],
    bins=bins,
    labels=labels
)

result = pd.DataFrame({
    "count": pending_group.value_counts(sort=False),
    "rate": pending_group.value_counts(normalize=True, sort=False) * 100
})

print("\n=== pending_votes 구간별 분포 ===")
print(result.round(2))

=== pending_votes 기본 통계 ===
count    677080.000000
mean         84.631143
std         123.262928
min           0.000000
25%           2.000000
50%          29.000000
75%         122.000000
max        3352.000000
Name: pending_votes, dtype: float64

=== pending_votes 주요 분위수 ===
0.50     29.0
0.75    122.0
0.90    249.0
0.95    339.0
0.99    544.0
Name: pending_votes, dtype: float64

=== pending_votes 구간별 분포 ===
                count   rate
pending_votes               
0              112793  16.66
1-20           195162  28.82
21-100         171760  25.37
101-500        187936  27.76
500+             9429   1.39


In [31]:
print("answer_status")
print(question_record_df["answer_status"].value_counts(dropna=False))

print("\nis_voted")
print(question_piece_df["is_voted"].value_counts(dropna=False))

print("\nis_skipped")
print(question_piece_df["is_skipped"].value_counts(dropna=False))

answer_status
answer_status
N    1097932
A     111761
P       7865
Name: count, dtype: int64

is_voted
is_voted
1    1217560
0      46789
Name: count, dtype: int64

is_skipped
is_skipped
0    1264349
Name: count, dtype: int64


In [32]:
payment_df = pd.read_csv(
    PROJECT_ROOT / "data" / "interim" / "accounts_paymenthistory_preprocessed.csv"
)

print(payment_df.shape)
print(payment_df.columns.tolist())
payment_df.head()

(95140, 5)
['id', 'productId', 'phone_type', 'created_at', 'user_id']


,id,productId,phone_type,created_at,user_id
0,6,heart.777,A,2023-05-13 21:28:34,1211127
1,7,heart.777,A,2023-05-13 21:29:39,1151343
2,8,heart.777,A,2023-05-13 21:31:33,1002147
3,9,heart.777,A,2023-05-13 21:31:39,1095040
4,11,heart.777,A,2023-05-13 21:34:32,1164081


In [33]:
payment_df["created_at"] = pd.to_datetime(
    payment_df["created_at"],
    errors="coerce"
)

In [34]:
# 전체 결제 건수
print("전체 결제 건수:", f"{len(payment_df):,}")

# 결제 사용자 수
n_payers = payment_df["user_id"].nunique()
print("결제 사용자 수:", f"{n_payers:,}")

# 전체 사용자 대비 결제 사용자 비율
n_users = user_df["id"].nunique()
print("전체 사용자 수:", f"{n_users:,}")
print("결제 사용자 비율:", f"{n_payers / n_users * 100:.2f}%")

# 사용자별 결제 횟수
payment_user = (
    payment_df
    .groupby("user_id")
    .size()
    .reset_index(name="payment_count")
)

print("\n=== 사용자별 결제 횟수 ===")
print(payment_user["payment_count"].describe())

# 1회 구매 / 반복 구매
payment_user["is_repeat"] = payment_user["payment_count"] >= 2

print("\n1회 구매 사용자:",
      f"{(~payment_user['is_repeat']).sum():,}",
      f"({(~payment_user['is_repeat']).mean()*100:.2f}%)")

print("2회 이상 구매 사용자:",
      f"{payment_user['is_repeat'].sum():,}",
      f"({payment_user['is_repeat'].mean()*100:.2f}%)")

# 상품별 결제
print("\n=== 상품별 결제 건수 ===")
print(payment_df["productId"].value_counts(dropna=False))

전체 결제 건수: 95,140
결제 사용자 수: 59,192
전체 사용자 수: 677,080
결제 사용자 비율: 8.74%

=== 사용자별 결제 횟수 ===
count    59192.000000
mean         1.607312
std          1.560483
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max         60.000000
Name: payment_count, dtype: float64

1회 구매 사용자: 43,049 (72.73%)
2회 이상 구매 사용자: 16,143 (27.27%)

=== 상품별 결제 건수 ===
productId
heart.777     57873
heart.1000    19309
heart.200     15822
heart.4000     2136
Name: count, dtype: int64


In [35]:
event_df = pd.read_csv(
    PROJECT_ROOT / "data" / "interim" / "hackle_events_preprocessed.csv"
)

property_df = pd.read_csv(
    PROJECT_ROOT / "data" / "interim" / "hackle_properties_preprocessed.csv"
)

print(event_df.shape)
print(event_df.columns.tolist())

print(property_df.shape)
print(property_df.columns.tolist())

(11441319, 10)
['event_id', 'event_datetime', 'event_key', 'session_id', 'item_name', 'page_name', 'friend_count', 'votes_count', 'heart_balance', 'question_id']
(327431, 8)
['id', 'session_id', 'user_id', 'language', 'osname', 'osversion', 'versionname', 'device_id']


In [36]:
# 사용자, 결제시간 순으로 정렬
payment_sorted = payment_df.sort_values(
    ["user_id", "created_at"]
).copy()

# 직전 결제 시각
payment_sorted["prev_payment_at"] = (
    payment_sorted
    .groupby("user_id")["created_at"]
    .shift(1)
)

# 직전 결제와의 시간 차이 (시간 단위)
payment_sorted["payment_gap_hours"] = (
    payment_sorted["created_at"] -
    payment_sorted["prev_payment_at"]
).dt.total_seconds() / 3600

# 첫 결제를 제외한 재결제 간격 분포
gap = payment_sorted["payment_gap_hours"].dropna()

print("=== 재결제 간격 ===")
print(gap.describe())

print("\n1시간 이내:",
      f"{(gap <= 1).sum():,}",
      f"({(gap <= 1).mean()*100:.2f}%)")

print("24시간 이내:",
      f"{(gap <= 24).sum():,}",
      f"({(gap <= 24).mean()*100:.2f}%)")

print("7일 이내:",
      f"{(gap <= 24*7).sum():,}",
      f"({(gap <= 24*7).mean()*100:.2f}%)")

=== 재결제 간격 ===
count    35948.000000
mean       103.442699
std        427.368906
min          0.000000
25%          0.042431
50%          6.577639
75%         46.751597
max       8649.825556
Name: payment_gap_hours, dtype: float64

1시간 이내: 13,592 (37.81%)
24시간 이내: 23,428 (65.17%)
7일 이내: 32,473 (90.33%)


In [37]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================================================
# 0. 데이터 로드
# =========================================================

# PROJECT_ROOT가 이미 있다면 그대로 사용
# 없다면 현재 notebook 위치에 맞게 수정
# PROJECT_ROOT = Path.cwd().parents[2]

DATA_DIR = PROJECT_ROOT / "data" / "interim"

user_df = pd.read_csv(DATA_DIR / "accounts_user_preprocessed.csv")
friend_df = pd.read_csv(DATA_DIR / "accounts_friendrequest_preprocessed.csv")
question_record_df = pd.read_csv(DATA_DIR / "accounts_userquestionrecord_preprocessed.csv")
payment_df = pd.read_csv(DATA_DIR / "accounts_paymenthistory_preprocessed.csv")

print("=== 데이터 로드 완료 ===")
print("user:", user_df.shape)
print("friend:", friend_df.shape)
print("question_record:", question_record_df.shape)
print("payment:", payment_df.shape)


# =========================================================
# 1. 날짜 타입 변환
# =========================================================

user_df["created_at"] = pd.to_datetime(user_df["created_at"], errors="coerce")

friend_df["created_at"] = pd.to_datetime(friend_df["created_at"], errors="coerce")
friend_df["updated_at"] = pd.to_datetime(friend_df["updated_at"], errors="coerce")

payment_df["created_at"] = pd.to_datetime(payment_df["created_at"], errors="coerce")

if "created_at" in question_record_df.columns:
    question_record_df["created_at"] = pd.to_datetime(
        question_record_df["created_at"],
        errors="coerce"
    )


# =========================================================
# 2. 전체 사용자 기준 베이스 테이블 생성
# =========================================================

base = user_df[["id", "pending_votes"]].copy()
base = base.rename(columns={"id": "user_id"})

TOTAL_USERS = base["user_id"].nunique()

print("\n전체 사용자 수:", f"{TOTAL_USERS:,}")


# =========================================================
# 3. 결제 여부 생성
# =========================================================

payer_ids = set(payment_df["user_id"].dropna().unique())

base["is_payer"] = (
    base["user_id"]
    .isin(payer_ids)
    .astype(int)
)

print(
    "결제 사용자:",
    f"{base['is_payer'].sum():,}",
    f"({base['is_payer'].mean() * 100:.2f}%)"
)


# =========================================================
# 4. 임시 잔존 지표 생성
# =========================================================
# 팀 분석에서 사용했던 기준:
# 2023-07-01 이후 Accepted 친구 관계 업데이트 또는 결제 기록이 존재하면 잔존
#
# 주의:
# 이것은 완전한 서비스 활동 로그 기반 retention은 아니므로
# 최종 발표에서는 "잔존 proxy"라고 표현하는 것이 안전함.
# =========================================================

RETENTION_CUTOFF = pd.Timestamp("2023-07-01")

# Accepted 친구관계의 사용자별 마지막 활동일
accepted = friend_df.loc[
    friend_df["status"] == "A",
    ["send_user_id", "receive_user_id", "updated_at"]
].copy()

friend_send_last = (
    accepted
    .groupby("send_user_id")["updated_at"]
    .max()
    .rename("friend_send_last")
)

friend_receive_last = (
    accepted
    .groupby("receive_user_id")["updated_at"]
    .max()
    .rename("friend_receive_last")
)

# 결제 마지막 일자
payment_last = (
    payment_df
    .groupby("user_id")["created_at"]
    .max()
    .rename("payment_last")
)

activity = pd.DataFrame(index=base["user_id"].unique())

activity = activity.join(friend_send_last, how="left")
activity = activity.join(friend_receive_last, how="left")
activity = activity.join(payment_last, how="left")

activity["last_activity"] = activity[
    ["friend_send_last", "friend_receive_last", "payment_last"]
].max(axis=1)

activity["is_retained"] = (
    activity["last_activity"] >= RETENTION_CUTOFF
).astype(int)

activity = activity.reset_index().rename(columns={"index": "user_id"})

base = base.merge(
    activity[["user_id", "is_retained", "last_activity"]],
    on="user_id",
    how="left"
)

print(
    "\n잔존 proxy 사용자:",
    f"{base['is_retained'].sum():,}",
    f"({base['is_retained'].mean() * 100:.2f}%)"
)


# =========================================================
# 공통 출력 함수
# =========================================================

def summarize_group(df, group_col, label):
    result = (
        df
        .groupby(group_col, observed=False)
        .agg(
            users=("user_id", "nunique"),
            payment_rate=("is_payer", "mean"),
            retention_rate=("is_retained", "mean")
        )
        .reset_index()
    )

    result["user_share"] = (
        result["users"] / result["users"].sum() * 100
    )

    result["payment_rate"] *= 100
    result["retention_rate"] *= 100

    print(f"\n{'=' * 70}")
    print(label)
    print("=" * 70)

    print(
        result.round({
            "user_share": 2,
            "payment_rate": 2,
            "retention_rate": 2
        }).to_string(index=False)
    )

    return result


# =========================================================
# ① 친구 연결
# =========================================================
# Accepted 관계 기준 사용자별 친구 연결량 계산
# 발신 + 수신 기준으로 사용자에게 연결된 Accepted 기록 수 계산
# =========================================================

sent_acc = (
    accepted
    .groupby("send_user_id")
    .size()
    .rename("sent_accepted")
)

received_acc = (
    accepted
    .groupby("receive_user_id")
    .size()
    .rename("received_accepted")
)

friend_summary = pd.DataFrame(index=base["user_id"].unique())

friend_summary = friend_summary.join(sent_acc, how="left")
friend_summary = friend_summary.join(received_acc, how="left")

friend_summary = friend_summary.fillna(0)

friend_summary["accepted_count"] = (
    friend_summary["sent_accepted"]
    + friend_summary["received_accepted"]
)

friend_summary = friend_summary.reset_index().rename(
    columns={"index": "user_id"}
)

friend_analysis = base.merge(
    friend_summary[["user_id", "accepted_count"]],
    on="user_id",
    how="left"
)

friend_analysis["accepted_count"] = (
    friend_analysis["accepted_count"]
    .fillna(0)
)

friend_analysis["friend_group"] = pd.cut(
    friend_analysis["accepted_count"],
    bins=[-1, 0, 9, 29, 59, 99, np.inf],
    labels=[
        "0",
        "1-9",
        "10-29",
        "30-59",
        "60-99",
        "100+"
    ]
)

friend_result = summarize_group(
    friend_analysis,
    "friend_group",
    "① 친구 연결 — Accepted 관계 수에 따른 결제율/잔존율"
)

print("\n[친구 연결 핵심]")
print(
    "Accepted 관계 0명 사용자:",
    f"{(friend_analysis['accepted_count'] == 0).sum():,}",
    f"({(friend_analysis['accepted_count'] == 0).mean() * 100:.2f}%)"
)

print(
    "Accepted 관계 중앙값:",
    friend_analysis["accepted_count"].median()
)


# =========================================================
# ② 질문 참여·콘텐츠
# =========================================================
# 사용자별 질문 기록 중 unread 비율 계산
#
# 임의로 한 그룹을 '문제'라고 선언하지 않고
# unread_rate 구간별 KPI 차이를 먼저 확인
# =========================================================

question_user = (
    question_record_df
    .groupby("user_id")
    .agg(
        total_questions=("id", "count"),
        unread_questions=("has_read", lambda x: (x == 0).sum())
    )
    .reset_index()
)

question_user["unread_rate"] = (
    question_user["unread_questions"]
    / question_user["total_questions"]
)

question_analysis = base.merge(
    question_user,
    on="user_id",
    how="inner"
)

question_analysis["unread_group"] = pd.cut(
    question_analysis["unread_rate"],
    bins=[-0.001, 0, 0.25, 0.50, 0.75, 1.0],
    labels=[
        "0%",
        "0-25%",
        "25-50%",
        "50-75%",
        "75-100%"
    ],
    include_lowest=True
)

question_result = summarize_group(
    question_analysis,
    "unread_group",
    "② 질문 참여 — 사용자별 미열람 비율에 따른 결제율/잔존율"
)

print("\n[질문 데이터 커버리지]")
print(
    "질문 기록 보유 사용자:",
    f"{question_analysis['user_id'].nunique():,}",
)

print(
    "전체 사용자 대비:",
    f"{question_analysis['user_id'].nunique() / TOTAL_USERS * 100:.2f}%"
)

print(
    "전체 질문 기록 기준 미열람률:",
    f"{(question_record_df['has_read'] == 0).mean() * 100:.2f}%"
)


# =========================================================
# ③ 투표 수신·반응
# =========================================================
# accounts_user.pending_votes 사용
# 전체 사용자 기반이므로 상세 투표 테이블보다 커버리지 안정적
# =========================================================

vote_analysis = base.copy()

vote_analysis["pending_group"] = pd.cut(
    vote_analysis["pending_votes"],
    bins=[-1, 0, 20, 100, 500, np.inf],
    labels=[
        "0",
        "1-20",
        "21-100",
        "101-500",
        "500+"
    ]
)

vote_result = summarize_group(
    vote_analysis,
    "pending_group",
    "③ 투표 수신·반응 — 미확인 투표 수에 따른 결제율/잔존율"
)

print("\n[미확인 투표 핵심]")
print("중앙값:", vote_analysis["pending_votes"].median())

high_pending = vote_analysis["pending_votes"] > 100

print(
    "100개 초과 사용자:",
    f"{high_pending.sum():,}",
    f"({high_pending.mean() * 100:.2f}%)"
)


# =========================================================
# ④ 상점·첫 구매
# =========================================================
# 장기 DB에는 shop 방문 여부가 없으므로
# 우선 '전체 사용자 → 구매 경험' 수준을 확인
#
# 실제 '상점 방문 → 구매' 전환은 Hackle 24일 데이터를 사용해야 함.
# =========================================================

print("\n" + "=" * 70)
print("④ 첫 구매 — 전체 서비스 DB")
print("=" * 70)

print(
    "전체 사용자:",
    f"{TOTAL_USERS:,}"
)

print(
    "결제 경험 사용자:",
    f"{base['is_payer'].sum():,}"
)

print(
    "전체 사용자 대비 결제 경험률:",
    f"{base['is_payer'].mean() * 100:.2f}%"
)


# point 구간별 결제율
user_payment_analysis = user_df[
    ["id", "point"]
].copy()

user_payment_analysis = user_payment_analysis.rename(
    columns={"id": "user_id"}
)

user_payment_analysis["is_payer"] = (
    user_payment_analysis["user_id"]
    .isin(payer_ids)
    .astype(int)
)

user_payment_analysis["point_group"] = pd.cut(
    user_payment_analysis["point"],
    bins=[-1, 99, 999, 4999, np.inf],
    labels=[
        "0-99",
        "100-999",
        "1000-4999",
        "5000+"
    ]
)

point_payment = (
    user_payment_analysis
    .groupby("point_group", observed=False)
    .agg(
        users=("user_id", "count"),
        payment_rate=("is_payer", "mean")
    )
)

point_payment["user_share"] = (
    point_payment["users"]
    / len(user_payment_analysis)
    * 100
)

point_payment["payment_rate"] *= 100

print("\n[point 구간별 결제율]")
print(
    point_payment[
        ["users", "user_share", "payment_rate"]
    ].round(2)
)


# =========================================================
# ⑤ 구매 후·재구매
# =========================================================

payment_user = (
    payment_df
    .groupby("user_id")
    .size()
    .reset_index(name="payment_count")
)

payment_user["repeat_raw"] = (
    payment_user["payment_count"] >= 2
)

print("\n" + "=" * 70)
print("⑤ 구매 후·재구매")
print("=" * 70)

print(
    "결제 사용자:",
    f"{len(payment_user):,}"
)

print(
    "결제 기록 1건 사용자:",
    f"{(~payment_user['repeat_raw']).sum():,}",
    f"({(~payment_user['repeat_raw']).mean() * 100:.2f}%)"
)

print(
    "결제 기록 2건 이상 사용자:",
    f"{payment_user['repeat_raw'].sum():,}",
    f"({payment_user['repeat_raw'].mean() * 100:.2f}%)"
)

print("\n주의: 위 값은 아직 '진짜 재구매율'이 아니라 결제 기록 횟수 기준")


# ---------------------------------------------------------
# 연속 결제 여부 확인
# ---------------------------------------------------------

payment_sorted = payment_df.sort_values(
    ["user_id", "created_at"]
).copy()

payment_sorted["prev_payment"] = (
    payment_sorted
    .groupby("user_id")["created_at"]
    .shift()
)

payment_sorted["gap_hours"] = (
    payment_sorted["created_at"]
    - payment_sorted["prev_payment"]
).dt.total_seconds() / 3600

gap = payment_sorted["gap_hours"].dropna()

print("\n[재결제 간격]")
print(
    "1시간 이내:",
    f"{(gap <= 1).sum():,}",
    f"({(gap <= 1).mean() * 100:.2f}%)"
)

print(
    "24시간 이내:",
    f"{(gap <= 24).sum():,}",
    f"({(gap <= 24).mean() * 100:.2f}%)"
)

print(
    "7일 이내:",
    f"{(gap <= 24*7).sum():,}",
    f"({(gap <= 24*7).mean() * 100:.2f}%)"
)


# =========================================================
# 6. 데이터 커버리지 요약
# =========================================================

coverage = pd.DataFrame({
    "구간": [
        "① 친구 연결",
        "② 질문 참여",
        "③ 투표 수신",
        "④ 결제",
        "⑤ 재구매"
    ],
    "관측 사용자 수": [
        friend_analysis["user_id"].nunique(),
        question_analysis["user_id"].nunique(),
        vote_analysis["user_id"].nunique(),
        TOTAL_USERS,
        payment_user["user_id"].nunique()
    ]
})

coverage["전체 사용자 대비 커버리지(%)"] = (
    coverage["관측 사용자 수"]
    / TOTAL_USERS
    * 100
)

print("\n" + "=" * 70)
print("데이터 커버리지")
print("=" * 70)

print(
    coverage.round(2).to_string(index=False)
)


# =========================================================
# 7. 최종 참고 문구
# =========================================================

print("\n" + "=" * 70)
print("해석 시 주의")
print("=" * 70)

print("""
1. 친구/결제 데이터는 장기 서비스 DB 기반.
2. pending_votes는 accounts_user 전체 사용자 기반.
3. 질문 데이터는 실제 기록이 존재하는 사용자만 분석하므로 커버리지를 반드시 같이 봐야 함.
4. Hackle은 아직 위 비교표에 직접 넣지 않음.
   Hackle은 2023-07-18 ~ 2023-08-10, 24일간의 단기 행동 로그이므로
   최종 구간 선정 후 실제 행동 경로를 보조적으로 검증하는 용도로 활용하는 것이 안전함.
5. 잔존율은 Accepted 친구 업데이트 또는 결제 활동을 이용한 proxy이므로
   절대적인 '서비스 retention'으로 해석하지 말고 집단 간 상대 비교용으로 사용.
""")

=== 데이터 로드 완료 ===
user: (677081, 11)
friend: (17147175, 7)
question_record: (1217558, 12)
payment: (95140, 5)

전체 사용자 수: 677,081
결제 사용자: 59,192 (8.74%)

잔존 proxy 사용자: 72,665 (10.73%)

① 친구 연결 — Accepted 관계 수에 따른 결제율/잔존율
friend_group  users  payment_rate  retention_rate  user_share
           0  26802          0.86            0.23        3.96
         1-9  77233          3.50            8.93       11.41
       10-29 193768          7.19            9.97       28.62
       30-59 242727         10.29           10.91       35.85
       60-99 115891         12.50           13.68       17.12
        100+  20660         13.83           19.60        3.05

[친구 연결 핵심]
Accepted 관계 0명 사용자: 26,802 (3.96%)
Accepted 관계 중앙값: 34.0

② 질문 참여 — 사용자별 미열람 비율에 따른 결제율/잔존율
unread_group  users  payment_rate  retention_rate  user_share
          0%     46          2.17            4.35        0.95
       0-25%    581          8.26            6.71       11.98
      25-50%   2637          9.67            6.26       

In [38]:
import pandas as pd

data_path = '../../data/interim/' # 예시 경로
questionset_df = pd.read_csv(data_path + 'polls_questionset_preprocessed.csv')
userquestionrecord_df = pd.read_csv(data_path + 'accounts_userquestionrecord_preprocessed.csv')
paymenthistory_df = pd.read_csv(data_path + 'accounts_paymenthistory_preprocessed.csv')

set_users = questionset_df['user_id'].nunique()

voted_users = userquestionrecord_df['user_id'].nunique()

paid_users = paymenthistory_df['user_id'].nunique()

# 4. 결과 출력
print(f"1단계 (투표 세트 할당): {set_users:,}명")
print(f"2단계 (투표 실제 완료): {voted_users:,}명 (전환율: {voted_users/set_users*100:.1f}%)")
print(f"3단계 (최종 결제 완료): {paid_users:,}명 (전환율: {paid_users/voted_users*100:.1f}%)")

1단계 (투표 세트 할당): 4,965명
2단계 (투표 실제 완료): 4,849명 (전환율: 97.7%)
3단계 (최종 결제 완료): 59,192명 (전환율: 1220.7%)


In [39]:
import pandas as pd

data_path = '../../data/interim/'

questionset_df = pd.read_csv(
    data_path + 'polls_questionset_preprocessed.csv'
)
userquestionrecord_df = pd.read_csv(
    data_path + 'accounts_userquestionrecord_preprocessed.csv'
)
paymenthistory_df = pd.read_csv(
    data_path + 'accounts_paymenthistory_preprocessed.csv'
)

# -------------------------------------------------
# 1. 각 단계 사용자 집합
# -------------------------------------------------

set_user_ids = set(
    questionset_df['user_id']
    .dropna()
    .unique()
)

vote_user_ids = set(
    userquestionrecord_df['user_id']
    .dropna()
    .unique()
)

paid_user_ids = set(
    paymenthistory_df['user_id']
    .dropna()
    .unique()
)

# -------------------------------------------------
# 2. 동일 사용자 기준 단계별 교집합
# -------------------------------------------------

# 투표 세트를 받은 사용자 중 질문 기록이 있는 사용자
set_to_vote_ids = set_user_ids & vote_user_ids

# 위 사용자 중 실제 결제 경험까지 있는 사용자
set_vote_to_paid_ids = set_to_vote_ids & paid_user_ids

# -------------------------------------------------
# 3. 사용자 수
# -------------------------------------------------

set_users = len(set_user_ids)
voted_users = len(set_to_vote_ids)
paid_users = len(set_vote_to_paid_ids)

# -------------------------------------------------
# 4. 결과
# -------------------------------------------------

print(f"1단계 (투표 세트 사용자): {set_users:,}명")

print(
    f"2단계 (질문 기록 보유): {voted_users:,}명 "
    f"(전환율: {voted_users / set_users * 100:.1f}%)"
)

print(
    f"3단계 (결제 경험): {paid_users:,}명 "
    f"(전환율: {paid_users / voted_users * 100:.1f}%)"
)

print(
    f"전체 전환율: {paid_users / set_users * 100:.1f}%"
)

1단계 (투표 세트 사용자): 4,965명
2단계 (질문 기록 보유): 4,848명 (전환율: 97.6%)
3단계 (결제 경험): 410명 (전환율: 8.5%)
전체 전환율: 8.3%


In [40]:
print(questionset_df['status'].value_counts(dropna=False))
print(questionset_df[['question_piece_id_list', 'opening_time', 'status', 'user_id']].head(10))

status
F    152740
O      4399
C       566
Name: count, dtype: int64
                              question_piece_id_list         opening_time  \
0  [998588, 998589, 998590, 998591, 998592, 99859...  2023-04-28 12:28:07   
1  [998689, 998691, 998693, 998695, 998697, 99869...  2023-04-28 12:28:38   
2  [998688, 998690, 998692, 998694, 998696, 99869...  2023-04-28 12:28:38   
3  [998768, 998769, 998770, 998771, 998772, 99877...  2023-04-28 12:28:57   
4  [998808, 998809, 998810, 998811, 998813, 99881...  2023-04-28 12:29:04   
5  [998828, 998829, 998830, 998831, 998832, 99883...  2023-04-28 12:29:05   
6  [998918, 998919, 998920, 998921, 998922, 99892...  2023-04-28 12:32:16   
7  [998938, 998939, 998940, 998941, 998942, 99894...  2023-04-28 12:29:40   
8  [998948, 998949, 998950, 998951, 998952, 99895...  2023-04-28 12:29:43   
9  [999008, 999009, 999010, 999011, 999012, 99901...  2023-04-28 12:30:14   

  status  user_id  
0      F   849438  
1      F   847375  
2      F   849446  
3  

In [41]:
import pandas as pd

# =========================================================
# 0. 데이터 로드
# =========================================================

data_path = '../../data/interim/'

uq_df = pd.read_csv(
    data_path + 'accounts_userquestionrecord_preprocessed.csv'
)

print("전체 행 수:", f"{len(uq_df):,}")
print("전체 사용자 수:", f"{uq_df['user_id'].nunique():,}")

# =========================================================
# 1. 기본 상태값 확인
# =========================================================

print("\n=== status 분포 ===")
print(uq_df['status'].value_counts(dropna=False))

print("\n=== has_read 분포 ===")
print(uq_df['has_read'].value_counts(dropna=False))

print("\n=== answer_status 분포 ===")
print(uq_df['answer_status'].value_counts(dropna=False))


# =========================================================
# 2. 사용자 단위 집계
# =========================================================

user_question = (
    uq_df
    .groupby('user_id')
    .agg(
        question_record_count=('id', 'count'),
        read_count=('has_read', lambda x: (x == 1).sum()),
        unread_count=('has_read', lambda x: (x == 0).sum())
    )
    .reset_index()
)

# 질문 기록이 1건 이상 있는 사용자
stage_1_users = user_question['user_id'].nunique()

# 한 번이라도 읽은 사용자
stage_2_users = (
    user_question.loc[user_question['read_count'] > 0, 'user_id']
    .nunique()
)

# 모든 질문을 다 읽은 사용자
stage_3_users = (
    user_question.loc[user_question['unread_count'] == 0, 'user_id']
    .nunique()
)


# =========================================================
# 3. 퍼널 출력
# =========================================================

print("\n" + "=" * 60)
print("질문 참여 퍼널")
print("=" * 60)

print(
    f"1단계 질문 기록 보유 사용자: "
    f"{stage_1_users:,}명"
)

print(
    f"2단계 1회 이상 질문 열람 사용자: "
    f"{stage_2_users:,}명 "
    f"(전환율: {stage_2_users / stage_1_users * 100:.2f}%)"
)

print(
    f"3단계 보유 질문 전체 열람 사용자: "
    f"{stage_3_users:,}명 "
    f"(전환율: {stage_3_users / stage_2_users * 100:.2f}%)"
)


# =========================================================
# 4. 사용자별 미열람 비율
# =========================================================

user_question['unread_rate'] = (
    user_question['unread_count']
    / user_question['question_record_count']
)

print("\n=== 사용자별 미열람 비율 요약 ===")
print(
    user_question['unread_rate']
    .describe(
        percentiles=[0.25, 0.5, 0.75, 0.9]
    )
)


# =========================================================
# 5. 미열람 비율 구간
# =========================================================

user_question['unread_group'] = pd.cut(
    user_question['unread_rate'],
    bins=[-0.001, 0, 0.25, 0.5, 0.75, 1.0],
    labels=[
        '0%',
        '0~25%',
        '25~50%',
        '50~75%',
        '75~100%'
    ],
    include_lowest=True
)

unread_summary = (
    user_question
    .groupby('unread_group', observed=False)
    .agg(
        users=('user_id', 'nunique'),
        avg_question_count=('question_record_count', 'mean')
    )
    .reset_index()
)

unread_summary['user_share'] = (
    unread_summary['users']
    / stage_1_users
    * 100
)

print("\n=== 미열람 비율 구간별 사용자 수 ===")
print(
    unread_summary.round({
        'avg_question_count': 2,
        'user_share': 2
    }).to_string(index=False)
)

전체 행 수: 1,217,558
전체 사용자 수: 4,849

=== status 분포 ===
status
C    1156322
I      60578
B        658
Name: count, dtype: int64

=== has_read 분포 ===
has_read
1    675931
0    541627
Name: count, dtype: int64

=== answer_status 분포 ===
answer_status
N    1097932
A     111761
P       7865
Name: count, dtype: int64

질문 참여 퍼널
1단계 질문 기록 보유 사용자: 4,849명
2단계 1회 이상 질문 열람 사용자: 4,816명 (전환율: 99.32%)
3단계 보유 질문 전체 열람 사용자: 46명 (전환율: 0.96%)

=== 사용자별 미열람 비율 요약 ===
count    4849.000000
mean        0.441706
std         0.184981
min         0.000000
25%         0.316129
50%         0.418182
75%         0.550000
90%         0.696763
max         1.000000
Name: unread_rate, dtype: float64

=== 미열람 비율 구간별 사용자 수 ===
unread_group  users  avg_question_count  user_share
          0%     46                9.80        0.95
       0~25%    581              177.46       11.98
      25~50%   2637              274.80       54.38
      50~75%   1251              264.75       25.80
     75~100%    334              174.07   

In [42]:
question_users = set(
    userquestionrecord_df['user_id']
    .dropna()
    .unique()
)

paid_users = set(
    paymenthistory_df['user_id']
    .dropna()
    .unique()
)

both_users = question_users & paid_users
paid_only_users = paid_users - question_users

print(f"질문 기록 사용자: {len(question_users):,}명")
print(f"결제 사용자: {len(paid_users):,}명")

print(
    f"질문 기록 + 결제 사용자: "
    f"{len(both_users):,}명"
)

print(
    f"질문 기록 없이 결제한 것으로 보이는 사용자: "
    f"{len(paid_only_users):,}명"
)

print(
    f"결제자 중 질문 기록 존재 비율: "
    f"{len(both_users) / len(paid_users) * 100:.2f}%"
)

질문 기록 사용자: 4,849명
결제 사용자: 59,192명
질문 기록 + 결제 사용자: 410명
질문 기록 없이 결제한 것으로 보이는 사용자: 58,782명
결제자 중 질문 기록 존재 비율: 0.69%


In [43]:
import pandas as pd

data_path = '../../data/interim/'

events = pd.read_csv(
    data_path + 'hackle_events_preprocessed.csv'
)

properties = pd.read_csv(
    data_path + 'hackle_properties_preprocessed.csv'
)

# --------------------------------------------------
# 1. 분석할 핵심 이벤트
# --------------------------------------------------

target_events = [
    'click_question_open',
    'view_shop',
    'click_purchase',
    'complete_purchase'
]

# --------------------------------------------------
# 2. session_id → user_id 연결
# --------------------------------------------------

# 세션 하나가 여러 행으로 존재할 수 있으므로
# session_id-user_id 조합만 남김
session_user = (
    properties[['session_id', 'user_id']]
    .dropna()
    .drop_duplicates()
)

# 이벤트에 user_id 연결
df = events.merge(
    session_user,
    on='session_id',
    how='inner'
)

# --------------------------------------------------
# 3. 핵심 이벤트만 필터링
# --------------------------------------------------

funnel_df = df[
    df['event_key'].isin(target_events)
].copy()

# --------------------------------------------------
# 4. 이벤트별 발생 건수 / 사용자 수
# --------------------------------------------------

summary = (
    funnel_df
    .groupby('event_key')
    .agg(
        event_count=('event_id', 'count'),
        user_count=('user_id', 'nunique'),
        session_count=('session_id', 'nunique')
    )
    .reindex(target_events)
)

print("=== Hackle 핵심 행동 현황 ===")
print(summary)

# --------------------------------------------------
# 5. 전체 Hackle 사용자 수
# --------------------------------------------------

total_hackle_users = df['user_id'].nunique()

print(f"\nHackle 전체 사용자 수: {total_hackle_users:,}명")

summary['user_coverage'] = (
    summary['user_count'] / total_hackle_users * 100
)

print("\n=== Hackle 전체 사용자 대비 비율 ===")
print(summary[['user_count', 'user_coverage']].round(2))

=== Hackle 핵심 행동 현황 ===
                     event_count  user_count  session_count
event_key                                                  
click_question_open       804390      110949         111805
view_shop                  26176       17122          17190
click_purchase             12858        6361           6384
complete_purchase           2181        1617           1619

Hackle 전체 사용자 수: 226,283명

=== Hackle 전체 사용자 대비 비율 ===
                     user_count  user_coverage
event_key                                     
click_question_open      110949          49.03
view_shop                 17122           7.57
click_purchase             6361           2.81
complete_purchase          1617           0.71


In [44]:
# =========================================================
# Hackle 사용자 기반 순차 퍼널
# =========================================================

# 각 이벤트 사용자 집합
question_open_users = set(
    funnel_df.loc[
        funnel_df['event_key'] == 'click_question_open',
        'user_id'
    ].unique()
)

shop_users = set(
    funnel_df.loc[
        funnel_df['event_key'] == 'view_shop',
        'user_id'
    ].unique()
)

purchase_click_users = set(
    funnel_df.loc[
        funnel_df['event_key'] == 'click_purchase',
        'user_id'
    ].unique()
)

purchase_complete_users = set(
    funnel_df.loc[
        funnel_df['event_key'] == 'complete_purchase',
        'user_id'
    ].unique()
)


# ---------------------------------------------------------
# 단계별 교집합
# ---------------------------------------------------------

# 질문을 연 사용자 중 상점 방문
q_to_shop = question_open_users & shop_users

# 질문 → 상점까지 경험한 사용자 중 상품 클릭
q_shop_to_click = q_to_shop & purchase_click_users

# 질문 → 상점 → 상품 클릭까지 경험한 사용자 중 구매 완료
q_shop_click_to_complete = (
    q_shop_to_click & purchase_complete_users
)


# ---------------------------------------------------------
# 결과 출력
# ---------------------------------------------------------

print("=== Hackle 사용자 기준 퍼널 ===\n")

print(
    f"1. 받은 질문 열기: "
    f"{len(question_open_users):,}명"
)

print(
    f"2. 이후 상점 이용 경험: "
    f"{len(q_to_shop):,}명 "
    f"({len(q_to_shop) / len(question_open_users) * 100:.2f}%)"
)

print(
    f"3. 상품 클릭 경험: "
    f"{len(q_shop_to_click):,}명 "
    f"({len(q_shop_to_click) / len(q_to_shop) * 100:.2f}%)"
)

print(
    f"4. 구매 완료 경험: "
    f"{len(q_shop_click_to_complete):,}명 "
    f"({len(q_shop_click_to_complete) / len(q_shop_to_click) * 100:.2f}%)"
)

print(
    f"\n질문 열기 → 구매 완료 전체 비율: "
    f"{len(q_shop_click_to_complete) / len(question_open_users) * 100:.2f}%"
)

=== Hackle 사용자 기준 퍼널 ===

1. 받은 질문 열기: 110,949명
2. 이후 상점 이용 경험: 14,802명 (13.34%)
3. 상품 클릭 경험: 5,135명 (34.69%)
4. 구매 완료 경험: 1,241명 (24.17%)

질문 열기 → 구매 완료 전체 비율: 1.12%


In [45]:
import pandas as pd

# df는 이전에 hackle_events + hackle_properties를
# session_id 기준으로 연결해서 user_id까지 붙인 데이터라고 가정

target_events = [
    'click_question_open',
    'view_shop',
    'click_purchase',
    'complete_purchase'
]

funnel = df[
    df['event_key'].isin(target_events)
].copy()

# 날짜 타입 변환
funnel['event_datetime'] = pd.to_datetime(
    funnel['event_datetime'],
    errors='coerce'
)

funnel = funnel.dropna(
    subset=['user_id', 'event_datetime']
)

# 시간순 정렬
funnel = funnel.sort_values(
    ['user_id', 'event_datetime']
)


# =========================================================
# 사용자별 실제 시간순 퍼널 탐색
# =========================================================

results = []

for user_id, g in funnel.groupby('user_id'):

    # 1. 질문 열기
    q_times = g.loc[
        g['event_key'] == 'click_question_open',
        'event_datetime'
    ]

    if q_times.empty:
        continue

    q_time = q_times.min()

    # 2. 질문 열기 이후 상점 방문
    shop_times = g.loc[
        (g['event_key'] == 'view_shop') &
        (g['event_datetime'] > q_time),
        'event_datetime'
    ]

    shop_time = shop_times.min() if not shop_times.empty else pd.NaT

    # 3. 상점 방문 이후 상품 클릭
    click_time = pd.NaT

    if pd.notna(shop_time):
        click_times = g.loc[
            (g['event_key'] == 'click_purchase') &
            (g['event_datetime'] > shop_time),
            'event_datetime'
        ]

        if not click_times.empty:
            click_time = click_times.min()

    # 4. 상품 클릭 이후 구매 완료
    complete_time = pd.NaT

    if pd.notna(click_time):
        complete_times = g.loc[
            (g['event_key'] == 'complete_purchase') &
            (g['event_datetime'] > click_time),
            'event_datetime'
        ]

        if not complete_times.empty:
            complete_time = complete_times.min()

    results.append({
        'user_id': user_id,
        'question_open_time': q_time,
        'shop_time': shop_time,
        'purchase_click_time': click_time,
        'purchase_complete_time': complete_time
    })


funnel_users = pd.DataFrame(results)


# =========================================================
# 단계별 사용자 수
# =========================================================

stage1 = len(funnel_users)

stage2 = funnel_users['shop_time'].notna().sum()

stage3 = funnel_users['purchase_click_time'].notna().sum()

stage4 = funnel_users['purchase_complete_time'].notna().sum()


print("=== Hackle 실제 시간순 퍼널 ===\n")

print(
    f"1. 받은 질문 열기: "
    f"{stage1:,}명"
)

print(
    f"2. 질문 열기 이후 상점 방문: "
    f"{stage2:,}명 "
    f"({stage2 / stage1 * 100:.2f}%)"
)

print(
    f"3. 상점 방문 이후 상품 클릭: "
    f"{stage3:,}명 "
    f"({stage3 / stage2 * 100:.2f}%)"
)

print(
    f"4. 상품 클릭 이후 구매 완료: "
    f"{stage4:,}명 "
    f"({stage4 / stage3 * 100:.2f}%)"
)

print(
    f"\n질문 열기 → 구매 완료 전체 전환율: "
    f"{stage4 / stage1 * 100:.2f}%"
)

=== Hackle 실제 시간순 퍼널 ===

1. 받은 질문 열기: 110,949명
2. 질문 열기 이후 상점 방문: 13,232명 (11.93%)
3. 상점 방문 이후 상품 클릭: 4,363명 (32.97%)
4. 상품 클릭 이후 구매 완료: 1,005명 (23.03%)

질문 열기 → 구매 완료 전체 전환율: 0.91%


In [49]:
import pandas as pd

# 데이터 경로
data_path = '../../data/interim/'

userquestionrecord_df = pd.read_csv(
    data_path + 'accounts_userquestionrecord_preprocessed.csv'
)

paymenthistory_df = pd.read_csv(
    data_path + 'accounts_paymenthistory_preprocessed.csv'
)

user_df = pd.read_csv(
    data_path + 'accounts_user_preprocessed.csv'
)

# 날짜 변환
paymenthistory_df['created_at'] = pd.to_datetime(
    paymenthistory_df['created_at'],
    errors='coerce'
)

user_df['created_at'] = pd.to_datetime(
    user_df['created_at'],
    errors='coerce'
)

# --------------------------------------------------
# 1. 질문 기록 없는 결제 사용자 추출
# --------------------------------------------------

question_users = set(
    userquestionrecord_df['user_id']
    .dropna()
    .unique()
)

paid_users = set(
    paymenthistory_df['user_id']
    .dropna()
    .unique()
)

paid_without_question = paid_users - question_users

paid_no_question_df = paymenthistory_df[
    paymenthistory_df['user_id'].isin(paid_without_question)
].copy()

print("질문 기록 없는 결제 사용자:",
      f"{paid_no_question_df['user_id'].nunique():,}명")

print("결제 기록 수:",
      f"{len(paid_no_question_df):,}건")


# --------------------------------------------------
# 2. 월별 결제 사용자 / 결제 건수
# --------------------------------------------------

paid_no_question_df['payment_month'] = (
    paid_no_question_df['created_at']
    .dt.to_period('M')
)

monthly_summary = (
    paid_no_question_df
    .groupby('payment_month')
    .agg(
        payment_users=('user_id', 'nunique'),
        payment_count=('user_id', 'size')
    )
    .reset_index()
)

print("\n=== 월별 결제 현황 ===")
print(monthly_summary.to_string(index=False))


# --------------------------------------------------
# 3. 사용자별 첫 결제 시점
# --------------------------------------------------

first_payment = (
    paid_no_question_df
    .groupby('user_id')['created_at']
    .min()
    .reset_index(name='first_payment_at')
)

# 가입일 연결
signup = (
    user_df[['id', 'created_at']]
    .rename(
        columns={
            'id': 'user_id',
            'created_at': 'signup_at'
        }
    )
)

first_payment = first_payment.merge(
    signup,
    on='user_id',
    how='left'
)

# 가입 후 첫 결제까지 걸린 시간
first_payment['hours_to_first_payment'] = (
    first_payment['first_payment_at']
    - first_payment['signup_at']
).dt.total_seconds() / 3600


# --------------------------------------------------
# 4. 가입 후 얼마나 빨리 결제했는지
# --------------------------------------------------

print("\n=== 가입 → 첫 결제 소요시간 ===")

print(
    first_payment['hours_to_first_payment']
    .describe(
        percentiles=[0.25, 0.5, 0.75, 0.9]
    )
)

print("\n가입 후 1시간 이내:",
      f"{(first_payment['hours_to_first_payment'] <= 1).mean() * 100:.2f}%")

print("가입 후 24시간 이내:",
      f"{(first_payment['hours_to_first_payment'] <= 24).mean() * 100:.2f}%")

print("가입 후 7일 이내:",
      f"{(first_payment['hours_to_first_payment'] <= 24*7).mean() * 100:.2f}%")

print("가입 후 30일 이내:",
      f"{(first_payment['hours_to_first_payment'] <= 24*30).mean() * 100:.2f}%")

질문 기록 없는 결제 사용자: 58,782명
결제 기록 수: 94,497건

=== 월별 결제 현황 ===
payment_month  payment_users  payment_count
      2023-05          52280          81632
      2023-06           5567           7674
      2023-07           1637           2203
      2023-08           1049           1411
      2023-09            341            511
      2023-10            129            227
      2023-11            119            199
      2023-12            103            204
      2024-01             93            161
      2024-02             43             75
      2024-03             34             70
      2024-04             44            107
      2024-05             15             23

=== 가입 → 첫 결제 소요시간 ===
count    58782.000000
mean       232.063394
std        579.647064
min          0.015030
25%         25.750672
50%         89.105720
75%        195.941903
90%        435.719365
max       8941.099104
Name: hours_to_first_payment, dtype: float64

가입 후 1시간 이내: 2.45%
가입 후 24시간 이내: 23.16%
가입 후 7일 이내: 69.2

In [50]:
import pandas as pd

data_path = '../../data/interim/'

user_df = pd.read_csv(data_path + 'accounts_user_preprocessed.csv')
friend_df = pd.read_csv(data_path + 'accounts_friendrequest_preprocessed.csv')
question_df = pd.read_csv(data_path + 'accounts_userquestionrecord_preprocessed.csv')
payment_df = pd.read_csv(data_path + 'accounts_paymenthistory_preprocessed.csv')


# 1. 서비스 가입/진입
user_count = user_df['id'].nunique()

# 2. 친구 관계 형성
# Accepted 관계의 발신자 + 수신자 모두 포함
accepted = friend_df[friend_df['status'] == 'A']

friend_users = set(accepted['send_user_id'].dropna()) | \
               set(accepted['receive_user_id'].dropna())

# 3. 질문 관련 기록
question_users = question_df['user_id'].nunique()

# 4. 결제
payment_users = payment_df['user_id'].nunique()


print("=== 서비스 DB에서 확인 가능한 행동 단계 ===")
print(f"서비스 가입 사용자: {user_count:,}명")
print(f"Accepted 친구 관계 보유 사용자: {len(friend_users):,}명")
print(f"질문 관련 기록 사용자: {question_users:,}명")
print(f"결제 사용자: {payment_users:,}명")

print("\n※ 현재 선정된 서비스 DB에는 '상점 방문' 자체를 직접 기록한 테이블이 있는지 별도 확인 필요")

=== 서비스 DB에서 확인 가능한 행동 단계 ===
서비스 가입 사용자: 677,081명
Accepted 친구 관계 보유 사용자: 650,279명
질문 관련 기록 사용자: 4,849명
결제 사용자: 59,192명

※ 현재 선정된 서비스 DB에는 '상점 방문' 자체를 직접 기록한 테이블이 있는지 별도 확인 필요


In [ ]:
import pandas as pd

data_path = '../../data/interim/'

# 확인할 서비스 DB 파일
files = {
    'accounts_user': 'accounts_user_preprocessed.csv',
    'accounts_friendrequest': 'accounts_friendrequest_preprocessed.csv',
    'accounts_user_contacts': 'accounts_user_contacts_preprocessed.csv',
    'accounts_userquestionrecord': 'accounts_userquestionrecord_preprocessed.csv',
    'polls_question': 'polls_question_preprocessed.csv',
    'polls_questionpiece': 'polls_questionpiece_preprocessed.csv',
    'polls_questionset': 'polls_questionset_preprocessed.csv',
    'accounts_paymenthistory': 'accounts_paymenthistory_preprocessed.csv'
}

results = []

for table_name, file_name in files.items():
    df = pd.read_csv(data_path + file_name)

    # 날짜 후보 컬럼 찾기
    date_cols = [
        col for col in df.columns
        if col in ['created_at', 'updated_at', 'answer_updated_at', 'opening_time']
    ]

    if not date_cols:
        results.append({
            'table': table_name,
            'date_column': '없음',
            'min_date': None,
            'max_date': None,
            'rows': len(df),
            'users': df['user_id'].nunique() if 'user_id' in df.columns else None
        })
        continue

    for col in date_cols:
        converted = pd.to_datetime(df[col], errors='coerce')

        results.append({
            'table': table_name,
            'date_column': col,
            'min_date': converted.min(),
            'max_date': converted.max(),
            'rows': len(df),
            'users': df['user_id'].nunique() if 'user_id' in df.columns else None
        })

period_df = pd.DataFrame(results)

print("=== 서비스 DB 테이블별 기간 ===")
print(period_df.to_string(index=False))

=== 서비스 DB 테이블별 기간 ===
                      table       date_column                   min_date                   max_date     rows    users
              accounts_user        created_at 2023-03-29 05:18:56.162368 2024-05-09 08:31:17.710824   677081      NaN
     accounts_friendrequest        created_at 2023-04-17 18:29:11.000000 2024-05-09 09:21:47.000000 17147175 649072.0
     accounts_friendrequest        updated_at 2023-04-17 18:29:11.000000 2024-05-09 09:21:47.000000 17147175 649072.0
     accounts_user_contacts                없음                        NaT                        NaT     5063   5063.0
accounts_userquestionrecord        created_at 2023-04-28 12:27:49.000000 2024-05-08 01:36:18.000000  1217558   4849.0
accounts_userquestionrecord answer_updated_at 2023-04-28 12:27:49.000000 2024-05-08 01:36:18.000000  1217558   4849.0
             polls_question        created_at 2023-03-31 15:22:53.000000 2023-06-06 06:15:52.000000     5025      NaN
        polls_questionpiece      

---

---

---

### 구간 선정 EDA

In [81]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
DATA_DIR = PROJECT_ROOT / "data" / "processed"

user_df = pd.read_csv(DATA_DIR / "accounts_user_preprocessed_2.csv")
friend_df = pd.read_csv(DATA_DIR / "accounts_friendrequest_preprocessed_2.csv")
payment_df = pd.read_csv(DATA_DIR / "accounts_paymenthistory_preprocessed_2.csv")
question_df = pd.read_csv(DATA_DIR / "accounts_userquestionrecord_preprocessed_2.csv")
piece_df = pd.read_csv(DATA_DIR / "polls_questionpiece_preprocessed_2.csv")
point_df = pd.read_csv(DATA_DIR / "accounts_pointhistory_preprocessed_2.csv")

dfs = {
    "accounts_user": user_df,
    "accounts_friendrequest": friend_df,
    "accounts_paymenthistory": payment_df,
    "accounts_userquestionrecord": question_df,
    "polls_questionpiece": piece_df,
    "accounts_pointhistory": point_df
}

for name, df in dfs.items():
    print(f"\n[{name}]")
    print("shape:", df.shape)
    print("columns:", df.columns.tolist())


[accounts_user]
shape: (677075, 11)
columns: ['id', 'gender', 'point', 'is_push_on', 'created_at', 'ban_status', 'report_count', 'alarm_count', 'pending_chat', 'pending_votes', 'group_id']

[accounts_friendrequest]
shape: (7793140, 6)
columns: ['id', 'status', 'created_at', 'updated_at', 'receive_user_id', 'send_user_id']

[accounts_paymenthistory]
shape: (95137, 5)
columns: ['id', 'productId', 'phone_type', 'created_at', 'user_id']

[accounts_userquestionrecord]
shape: (800793, 12)
columns: ['id', 'status', 'created_at', 'chosen_user_id', 'question_id', 'user_id', 'question_piece_id', 'has_read', 'answer_status', 'answer_updated_at', 'report_count', 'opened_times']

[polls_questionpiece]
shape: (823739, 5)
columns: ['id', 'is_voted', 'created_at', 'question_id', 'is_skipped']

[accounts_pointhistory]
shape: (88762, 5)
columns: ['id', 'delta_point', 'created_at', 'user_id', 'user_question_record_id']


In [82]:
# 친구 관계가 형성된 사용자 추출
send_users = friend_df["send_user_id"]
receive_users = friend_df["receive_user_id"]

friend_users = pd.concat(
    [send_users, receive_users],
    ignore_index=True
).drop_duplicates()

# accounts_user 모집단 안에 있는 사용자만 유지
base_users = set(user_df["id"])
friend_users = set(friend_users) & base_users

base_count = user_df["id"].nunique()
friend_count = len(friend_users)

conversion_rate = friend_count / base_count * 100

print("전체 모집단:", base_count)
print("친구 관계 형성 사용자:", friend_count)
print(f"가입/진입 → 친구 관계 형성 전환율: {conversion_rate:.2f}%")

전체 모집단: 677075
친구 관계 형성 사용자: 574163
가입/진입 → 친구 관계 형성 전환율: 84.80%


In [84]:
# 투표를 받은 사용자
vote_received_users = set(
    question_df["chosen_user_id"]
    .dropna()
    .unique()
)

# 친구 관계 형성 사용자 중 투표를 받은 사용자
friend_to_vote_users = friend_users & vote_received_users

friend_count = len(friend_users)
vote_received_count = len(friend_to_vote_users)

conversion_rate_2 = (
    vote_received_count / friend_count * 100
)

print("친구 관계 형성 사용자:", friend_count)
print("친구 관계 형성 후 투표 받은 사용자:", vote_received_count)
print(
    f"친구 관계 형성 → 투표 수신 전환율: "
    f"{conversion_rate_2:.2f}%"
)

친구 관계 형성 사용자: 574163
친구 관계 형성 후 투표 받은 사용자: 10647
친구 관계 형성 → 투표 수신 전환율: 1.85%


In [83]:
# 결제 사용자
payment_users = set(
    payment_df["user_id"]
    .dropna()
    .unique()
)

# 투표 수신 단계 사용자
vote_stage_users = friend_to_vote_users

# 투표 수신 사용자 중 결제한 사용자
vote_to_payment_users = vote_stage_users & payment_users

vote_stage_count = len(vote_stage_users)
payment_count = len(vote_to_payment_users)

conversion_rate_3 = payment_count / vote_stage_count * 100

print("투표 수신 단계 사용자:", vote_stage_count)
print("결제 사용자:", payment_count)
print(f"투표 수신 → 결제 전환율: {conversion_rate_3:.2f}%")

투표 수신 단계 사용자: 10647
결제 사용자: 1041
투표 수신 → 결제 전환율: 9.78%


In [85]:
# 투표 수신 단계 사용자
vote_stage_users = friend_to_vote_users

# 재화 소모 사용자
spend_users = set(
    point_df["user_id"]
    .dropna()
    .unique()
)

# 투표 수신 사용자 중 재화를 소모한 사용자
vote_to_spend_users = vote_stage_users & spend_users

vote_stage_count = len(vote_stage_users)
spend_count = len(vote_to_spend_users)

conversion_rate_spend = (
    spend_count / vote_stage_count * 100
)

print("투표 수신 단계 사용자:", vote_stage_count)
print("재화 소모 사용자:", spend_count)
print(
    f"투표 수신 → 재화 소모 전환율: "
    f"{conversion_rate_spend:.2f}%"
)

투표 수신 단계 사용자: 10647
재화 소모 사용자: 3367
투표 수신 → 재화 소모 전환율: 31.62%


In [87]:
# 결제 사용자
payment_users = set(
    payment_df["user_id"]
    .dropna()
    .unique()
)

# 앞 단계까지 도달한 사용자
# 친구 관계 형성 + 투표 수신 + 재화 소모
spend_stage_users = vote_to_spend_users

# 재화 소모 사용자 중 결제한 사용자
spend_to_payment_users = spend_stage_users & payment_users

spend_stage_count = len(spend_stage_users)
payment_count = len(spend_to_payment_users)

conversion_rate_4 = (
    payment_count / spend_stage_count * 100
)

print("재화 소모 단계 사용자:", spend_stage_count)
print("결제 사용자:", payment_count)
print(
    f"재화 소모 → 결제 전환율: "
    f"{conversion_rate_4:.2f}%"
)

재화 소모 단계 사용자: 3367
결제 사용자: 358
재화 소모 → 결제 전환율: 10.63%


In [88]:
# 날짜형 변환
payment_df["created_at"] = pd.to_datetime(payment_df["created_at"])
question_df["created_at"] = pd.to_datetime(question_df["created_at"])

# 결제 단계까지 도달한 358명
payment_stage_users = spend_to_payment_users

# 358명의 결제 기록만 추출
payment_stage_df = payment_df[
    payment_df["user_id"].isin(payment_stage_users)
].copy()

# 사용자별 첫 결제 시점
first_payment = (
    payment_stage_df
    .groupby("user_id", as_index=False)["created_at"]
    .min()
    .rename(columns={
        "created_at": "first_payment_at"
    })
)

# 투표 수신 기록
vote_received_df = (
    question_df[
        ["chosen_user_id", "created_at"]
    ]
    .dropna(subset=["chosen_user_id"])
    .rename(columns={
        "chosen_user_id": "user_id",
        "created_at": "vote_received_at"
    })
)

# 첫 결제 정보와 투표 수신 기록 연결
after_payment_vote = first_payment.merge(
    vote_received_df,
    on="user_id",
    how="left"
)

# 첫 결제 이후 다시 투표를 받은 기록만
after_payment_vote = after_payment_vote[
    after_payment_vote["vote_received_at"]
    > after_payment_vote["first_payment_at"]
]

# 사용자 수
after_payment_vote_users = set(
    after_payment_vote["user_id"].unique()
)

print("결제 단계 사용자:", len(payment_stage_users))
print(
    "첫 결제 이후 다시 투표 수신한 사용자:",
    len(after_payment_vote_users)
)

결제 단계 사용자: 358
첫 결제 이후 다시 투표 수신한 사용자: 353


In [89]:
# 첫 결제 이후 투표 수신 기록 중 필요한 컬럼만
vote_after_payment_detail = after_payment_vote[
    ["user_id", "vote_received_at"]
].copy()

# 투표 기록과 포인트 소모 기록 연결
vote_spend = point_df.merge(
    question_df[
        ["id", "chosen_user_id", "created_at"]
    ],
    left_on="user_question_record_id",
    right_on="id",
    how="inner",
    suffixes=("_spend", "_vote")
)

# 포인트를 쓴 사용자와 투표를 받은 사용자가 같은 경우만
vote_spend = vote_spend[
    vote_spend["user_id"] == vote_spend["chosen_user_id"]
].copy()

# 첫 결제 시점 연결
vote_spend = vote_spend.merge(
    first_payment,
    on="user_id",
    how="inner"
)

# 첫 결제 이후 받은 투표 + 그 이후 재화 소모
vote_spend_after_payment = vote_spend[
    (vote_spend["created_at_vote"] > vote_spend["first_payment_at"]) &
    (vote_spend["created_at_spend"] >= vote_spend["created_at_vote"])
].copy()

# 사용자 수
vote_spend_users = set(
    vote_spend_after_payment["user_id"].unique()
)

print("첫 결제 이후 투표 수신 사용자:", len(after_payment_vote_users))
print(
    "이후 투표 수신 + 재화 소모 사용자:",
    len(vote_spend_users)
)

첫 결제 이후 투표 수신 사용자: 353
이후 투표 수신 + 재화 소모 사용자: 329


In [90]:
# 사용자별 첫 '투표 수신 + 재화 소모' 시점
first_vote_spend = (
    vote_spend_after_payment
    .groupby("user_id", as_index=False)["created_at_spend"]
    .min()
    .rename(columns={
        "created_at_spend": "first_vote_spend_at"
    })
)

# 해당 사용자들의 결제 기록 연결
repurchase_check = payment_df[
    ["user_id", "created_at"]
].merge(
    first_vote_spend,
    on="user_id",
    how="inner"
)

# 재화 소모 이후 다시 결제한 기록만
repurchase_check = repurchase_check[
    repurchase_check["created_at"] > repurchase_check["first_vote_spend_at"]
]

# 재결제 사용자
repurchase_users = set(
    repurchase_check["user_id"].unique()
)

repurchase_count = len(repurchase_users)

print("첫 결제 이후 투표 수신 + 재화 소모 사용자:", len(first_vote_spend))
print("재화 소모 이후 재결제 사용자:", repurchase_count)

# 2차 루프 기준 전환율
repurchase_rate = (
    repurchase_count / len(payment_stage_users) * 100
)

print(f"결제 → 재결제 전환율: {repurchase_rate:.2f}%")

첫 결제 이후 투표 수신 + 재화 소모 사용자: 329
재화 소모 이후 재결제 사용자: 68
결제 → 재결제 전환율: 18.99%


In [91]:
# 첫 결제 이후 다시 투표를 받은 사용자 353명
second_vote_users = after_payment_vote_users

# 353명의 첫 결제 시점
second_loop_base = first_payment[
    first_payment["user_id"].isin(second_vote_users)
].copy()

# 포인트 소모 기록 연결
spend_after_payment = point_df[
    ["user_id", "created_at"]
].merge(
    second_loop_base,
    on="user_id",
    how="inner"
)

# 첫 결제 이후 재화 소모
spend_after_payment = spend_after_payment[
    spend_after_payment["created_at"] >
    spend_after_payment["first_payment_at"]
]

# 재화 소모 사용자
second_spend_users = set(
    spend_after_payment["user_id"].unique()
)

print("첫 결제 이후 투표 수신 사용자:", len(second_vote_users))
print("이후 재화 소모 사용자:", len(second_spend_users))

첫 결제 이후 투표 수신 사용자: 353
이후 재화 소모 사용자: 347


In [92]:
# 사용자별 첫 결제 이후 첫 투표 수신 시점
first_vote_after_payment = (
    after_payment_vote
    .groupby("user_id", as_index=False)["vote_received_at"]
    .min()
    .rename(columns={
        "vote_received_at": "first_vote_after_payment_at"
    })
)

# 포인트 소모 기록 연결
spend_after_vote = point_df[
    ["user_id", "created_at"]
].merge(
    first_vote_after_payment,
    on="user_id",
    how="inner"
)

# 다시 투표를 받은 이후 발생한 재화 소모만
spend_after_vote = spend_after_vote[
    spend_after_vote["created_at"] >
    spend_after_vote["first_vote_after_payment_at"]
]

second_spend_users = set(
    spend_after_vote["user_id"].unique()
)

print("첫 결제 이후 투표 수신 사용자:", len(first_vote_after_payment))
print("투표 수신 이후 재화 소모 사용자:", len(second_spend_users))

첫 결제 이후 투표 수신 사용자: 353
투표 수신 이후 재화 소모 사용자: 334


In [93]:
# 사용자별 '재투표 수신 이후 첫 재화 소모 시점'
first_spend_after_vote = (
    spend_after_vote
    .groupby("user_id", as_index=False)["created_at"]
    .min()
    .rename(columns={
        "created_at": "first_spend_after_vote_at"
    })
)

# 결제 기록 연결
repurchase_check = payment_df[
    ["user_id", "created_at"]
].merge(
    first_spend_after_vote,
    on="user_id",
    how="inner"
)

# 재화 소모 이후 다시 결제한 기록만
repurchase_check = repurchase_check[
    repurchase_check["created_at"] >
    repurchase_check["first_spend_after_vote_at"]
]

# 재결제 사용자
repurchase_users = set(
    repurchase_check["user_id"].unique()
)

repurchase_count = len(repurchase_users)

print("재투표 수신 이후 재화 소모 사용자:", len(first_spend_after_vote))
print("재화 소모 이후 재결제 사용자:", repurchase_count)

# 최초 결제 단계 358명 기준 전체 재결제 전환율
repurchase_rate = (
    repurchase_count / len(payment_stage_users) * 100
)

print(f"결제 → 재결제 전체 전환율: {repurchase_rate:.2f}%")

재투표 수신 이후 재화 소모 사용자: 334
재화 소모 이후 재결제 사용자: 71
결제 → 재결제 전체 전환율: 19.83%
